# Lab: The Deutsch-Jozsa Algorithm with Qiskit

In this lab we implement the **Deutsch-Jozsa algorithm**, one of the first examples of a quantum algorithm that outperforms any classical deterministic algorithm.

We are given a black-box ("oracle") function:

$$f: \{0, 1\}^n \rightarrow \{0, 1\}$$

with the **promise** that $f$ is either:

- **constant**: $f(x)$ is the same for every input $x$, or
- **balanced**: $f(x) = 0$ for exactly half of the inputs and $f(x) = 1$ for the other half.

Our job is to decide whether $f$ is constant or balanced.

- A classical deterministic algorithm may need up to $2^{n-1} + 1$ oracle queries in the worst case.
- The Deutsch-Jozsa algorithm answers the question with certainty using **a single** oracle query.

We will build the algorithm in three tasks:
1. Implement a constant oracle and a balanced oracle.
2. Implement the circuit that runs the algorithm.
3. Implement the algorithm that runs the circuit and decides constant vs balanced.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

The oracles act on $n$ input qubits **plus one ancilla qubit** used for phase kickback. We adopt the convention that the ancilla is the last qubit, at index $n$.

The helper `oracle_value(oracle, n, x)` evaluates a classical oracle on the input `x` by simulating $U_f |x\rangle|0\rangle$ and reading the ancilla. This lets the test cells below check the oracles.

In [3]:
def validate_oracle_inputs(n: int, oracle: QuantumCircuit | None = None) -> None:
    """Guardrail: checks n is a positive integer and the oracle has n + 1 qubits."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    if oracle is not None:
        assert oracle.num_qubits == n + 1, (
            f"Oracle must act on n + 1 = {n + 1} qubits (n inputs plus one ancilla), "
            f"but it acts on {oracle.num_qubits}."
        )


def all_basis_states(n: int) -> list[str]:
    """Returns every n-bit computational basis state as a bitstring."""
    return [format(i, f'0{n}b') for i in range(2 ** n)]


def oracle_value(oracle: QuantumCircuit, n: int, x: str) -> int:
    """
    Evaluates a classical oracle f on the input x.

    Simulates U_f |x>|0>, where the ancilla is qubit n, and returns f(x).
    """
    validate_oracle_inputs(n, oracle)
    assert len(x) == n, f"Input '{x}' must have length n = {n}."
    assert set(x).issubset({'0', '1'}), f"Input '{x}' must be a binary string."

    circuit = QuantumCircuit(n + 1)
    for i, bit in enumerate(reversed(x)):
        if bit == '1':
            circuit.x(i)
    circuit.compose(oracle, inplace=True)

    statevector = Statevector(circuit)
    return int(round(statevector.probabilities([n])[1]))


print("Helper functions and guardrails loaded successfully!")

Helper functions and guardrails loaded successfully!


## Task 1: Building the Oracles

An oracle is a reversible circuit $U_f$ that acts on $|x\rangle|y\rangle$ as

$$U_f |x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle.$$

For the Deutsch-Jozsa algorithm the ancilla is prepared in $|-\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$, so that flipping the ancilla becomes a phase:

$$U_f |x\rangle |-\rangle = (-1)^{f(x)} |x\rangle |-\rangle.$$

We will implement two oracles, each on $n + 1$ qubits with the ancilla at index $n$.

### Task 1.1: Constant Oracles

Implement two constant oracles, each on `n` input qubits plus one ancilla:

- `constant_zero_oracle(n: int) -> QuantumCircuit`: implements $f(x) = 0$ for all $x$.
- `constant_one_oracle(n: int) -> QuantumCircuit`: implements $f(x) = 1$ for all $x$.

> Hint: a constant function either never flips the ancilla, or always flips it.

In [4]:
def constant_zero_oracle(n: int) -> QuantumCircuit:
    """
    Builds the constant-zero oracle f(x) = 0 on n input qubits plus one ancilla.

    Args:
        n: Number of input qubits.

    Returns:
        QuantumCircuit: An (n + 1)-qubit circuit implementing the oracle.
    """
    validate_oracle_inputs(n)

    # TODO: implement the constant-zero oracle
    qc = QuantumCircuit(n+1)
    
    return qc

In [48]:
def constant_one_oracle(n: int) -> QuantumCircuit:
    """
    Builds the constant-one oracle f(x) = 1 on n input qubits plus one ancilla.

    Args:
        n: Number of input qubits.

    Returns:
        QuantumCircuit: An (n + 1)-qubit circuit implementing the oracle.
    """
    validate_oracle_inputs(n)

    # TODO: implement the constant-one oracle
    qc = QuantumCircuit(n+1)
    qc.x(n)
    
    return qc

### Task 1.2: Balanced Oracle

Implement `balanced_oracle(n: int) -> QuantumCircuit`.

- `n`: number of input qubits.
- Returns an `(n + 1)`-qubit circuit implementing the balanced function
  $f(x) = x_0 \oplus x_1 \oplus \dots \oplus x_{n-1}$, where $\oplus$ is addition modulo 2.

> Hint: `f(x)` is the parity of the input bits. A CNOT from each input qubit to the ancilla accumulates this parity.

In [6]:
def balanced_oracle(n: int) -> QuantumCircuit:
    """
    Builds the balanced oracle f(x) = x_0 XOR x_1 XOR ... XOR x_{n-1}
    on n input qubits plus one ancilla.

    Args:
        n: Number of input qubits.

    Returns:
        QuantumCircuit: An (n + 1)-qubit circuit implementing the oracle.
    """
    validate_oracle_inputs(n)
    qc = QuantumCircuit(n+1)
    for i in range(n):
        if i%2==0:
            qc.x(i)
            qc.cx(i, n)


    # TODO: implement the balanced oracle
    return qc

### Task 1.3: Test the Oracles

The cells below check that your oracles implement the promised functions:

- The constant oracle must return `value` for **every** input.
- The balanced oracle must return `0` for exactly half of the inputs and `1` for the other half.

In [41]:
# Constant oracles: each must return its fixed value for every input
for n in [1, 2, 3]:
    zero_oracle = constant_zero_oracle(n)
    one_oracle = constant_one_oracle(n)
    for x in all_basis_states(n):
        assert oracle_value(zero_oracle, n, x) == 0, (
            f"Constant-zero oracle n={n} returned 1 for x={x}"
        )
        assert oracle_value(one_oracle, n, x) == 1, (
            f"Constant-one oracle n={n} returned 0 for x={x}"
        )
print("Constant oracles passed all tests!")

Constant oracles passed all tests!


In [42]:
# Balanced oracle: exactly half of the inputs map to 0 and half to 1
for n in [1, 2, 3]:
    oracle = balanced_oracle(n)
    values = [oracle_value(oracle, n, x) for x in all_basis_states(n)]
    assert set(values) == {0, 1}, f"Balanced oracle n={n} is not balanced: {values}"
    assert values.count(0) == values.count(1) == 2 ** (n - 1), (
        f"Balanced oracle n={n} has the wrong split: "
        f"{values.count(0)} zeros and {values.count(1)} ones"
    )
print("Balanced oracle passed all tests!")

Balanced oracle passed all tests!


## Task 2: Building the Algorithm Circuit

Implement `dj_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit`.

- `n`: number of input qubits.
- `oracle`: an `(n + 1)`-qubit oracle circuit, as built above.
- Returns a circuit implementing the Deutsch-Jozsa algorithm, with `n` classical bits measuring the input register.

The circuit you must build is shown below. The ancilla (the last qubit) provides phase kickback, and only the `n` input qubits are measured.

```
                      ┌───────┐
q_0:     |0> ─ ─ ─ H ─┤       ├─ H ─ M
q_1:     |0> ─ ─ ─ H ─┤       ├─ H ─ M
  ⋮             ⋮      │  U_f  │  ⋮   ⋮
q_(n-1): |0> ─ ─ ─ H ─┤       ├─ H ─ M
q_n:     |0> ─ X ─ H ─┤       ├─ H
                      └───────┘
```

In [52]:
def dj_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit:
    """
    Builds a circuit implementing the Deutsch-Jozsa algorithm.

    Args:
        n: Number of input qubits.
        oracle: An (n + 1)-qubit oracle circuit.

    Returns:
        QuantumCircuit: The Deutsch-Jozsa circuit with n classical bits
        measuring the input register.
    """
    validate_oracle_inputs(n, oracle)
    qc = QuantumCircuit(n+1, n)
    qc.x(n)
    for i in range(n+1):
        qc.h(i)
    qc = qc.compose(oracle)
    for i in range(n+1):
        qc.h(i)
    qc.measure(range(n), range(n))

    # TODO: implement the circuit shown in the diagram above
    return qc

## Task 3: Running the Algorithm

Implement `deutsch_jozsa(n: int, oracle: QuantumCircuit) -> bool`.

- `n`: number of input qubits.
- `oracle`: an `(n + 1)`-qubit oracle circuit.
- Returns `False` if `f` is **constant** and `True` if `f` is **balanced**.

With probability 1:

- If the oracle is **constant**, measuring the input register always yields $|0\cdots0\rangle$.
- If the oracle is **balanced**, the input register is never $|0\cdots0\rangle$.

> Hint: build the circuit with `dj_circuit`, run it on `AerSimulator`, and inspect `get_counts()`. If `'0' * n` appears, the function is constant.

In [57]:
def deutsch_jozsa(n: int, oracle: QuantumCircuit) -> bool:
    """
    Runs the Deutsch-Jozsa algorithm and decides whether the oracle is
    constant or balanced.

    Args:
        n: Number of input qubits.
        oracle: An (n + 1)-qubit oracle circuit.

    Returns:
        bool: False if the oracle is constant, True if it is balanced.
    """
    validate_oracle_inputs(n, oracle)
    qc = dj_circuit(n, oracle)
    simulator = AerSimulator()
    job = simulator.run(qc)
    result = job.result()
    counts = result.get_counts()
    return not '0'*n in counts

    # TODO: build the circuit with dj_circuit, run it on AerSimulator,
    # and return False for constant or True for balanced.

### Final Test

The cell below checks that your algorithm always returns the correct result: `False` for the constant oracles and `True` for the balanced oracle.

In [58]:
for n in [1, 2, 3]:
    for oracle in [constant_zero_oracle(n), constant_one_oracle(n)]:
        result = deutsch_jozsa(n, oracle)
        assert result is False, (
            f"Constant oracle n={n} should return False, got {result}"
        )

    result = deutsch_jozsa(n, balanced_oracle(n))
    assert result is True, (
        f"Balanced oracle n={n} should return True, got {result}"
    )

print("Deutsch-Jozsa passed all tests!")

Deutsch-Jozsa passed all tests!
